# LangGraph Topic 2 — **State** (Deep Practical Notes)

**Official docs section:** from the LangGraph Graph API / low-level concepts docs in [LangGraph documentation](https://docs.langchain.com/oss/python/langgraph/graph-api?utm_source=chatgpt.com)

---

# 1) What is State in LangGraph?

**State = the shared data object of the graph.**

It is the central object that flows through every node.

Every node in a LangGraph app does this:

1. **Receives the current state**
2. **Reads values from it**
3. **Returns updates to it**
4. **Passes the updated state forward**

So instead of passing many separate variables between functions, LangGraph keeps a **single shared state object**.

---

# 2) Mental Model

Think of state like a **shared notebook** passed from one worker to another.

Example flow:

```text
Initial State
{
  "question": "What is AI?",
  "answer": ""
}

      ↓

Node 1 reads question
Node 1 writes answer

      ↓

Updated State
{
  "question": "What is AI?",
  "answer": "Artificial Intelligence is ..."
}
```

So:

* **nodes = workers**
* **state = shared notebook**
* **edges = routing rules telling which worker gets it next**

---

# 3) Why LangGraph Needs State

Without state, nodes would have no clean way to exchange data.

Example workflow:

* user asks a question
* one node rewrites the question
* another node searches docs
* another node calls an LLM
* another node formats the final answer

All of these need access to shared information like:

* original question
* rewritten query
* retrieved docs
* final answer
* tool outputs
* chat history
* status flags
* error counters

That shared information lives in **state**.

---

# 4) Basic State Example

## Define state with `TypedDict`

```python
from typing_extensions import TypedDict

class State(TypedDict):
    question: str
    answer: str
```

This means the graph state contains:

* `question`
* `answer`

Example initial input:

```python
{
    "question": "What is machine learning?",
    "answer": ""
}
```

---

# 5) How Nodes Use State

Example node:

```python
def chatbot(state: State):
    question = state["question"]

    return {
        "answer": f"You asked: {question}"
    }
```

### What happens here?

Node receives:

```python
{
    "question": "What is machine learning?",
    "answer": ""
}
```

Node returns:

```python
{
    "answer": "You asked: What is machine learning?"
}
```

LangGraph merges that into state, so the new state becomes:

```python
{
    "question": "What is machine learning?",
    "answer": "You asked: What is machine learning?"
}
```

---

# 6) Important Rule: Nodes Return **State Updates**, Not the Whole State

This is one of the most important LangGraph concepts.

If your state is:

```python
class State(TypedDict):
    question: str
    answer: str
    step_count: int
```

and your node only changes `answer`, then it should return only that update:

```python
def chatbot(state: State):
    return {
        "answer": "Hello from LangGraph"
    }
```

Not this:

```python
def chatbot(state: State):
    return {
        "question": state["question"],
        "answer": "Hello from LangGraph",
        "step_count": state["step_count"]
    }
```

That second version is unnecessary unless you intentionally want to update those fields too.

### Best practice:

Return **only the keys you want to update**.


In [1]:
# 7) Full Minimal StateGraph Example

from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    message: str
    reply: str

def chatbot(state: State):
    return {
        "reply": f"Echo: {state['message']}"
    }

builder = StateGraph(State)

builder.add_node("chatbot", chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

graph = builder.compile()

result = graph.invoke({
    "message": "Hello",
    "reply": ""
})

print(result)

{'message': 'Hello', 'reply': 'Echo: Hello'}


### Output

```python
{
    "message": "Hello",
    "reply": "Echo: Hello"
}
```

---

# 8) State Lifecycle During Execution

Let’s track state step by step.

## Initial state

```python
{
    "message": "Hello",
    "reply": ""
}
```

## `chatbot` node receives it

```python
state["message"]   # "Hello"
state["reply"]     # ""
```

## Node returns

```python
{
    "reply": "Echo: Hello"
}
```

## LangGraph merges update into state

Final state:

```python
{
    "message": "Hello",
    "reply": "Echo: Hello"
}
```

---

# 9) State Is Shared Across All Nodes

Example:

```python
class State(TypedDict):
    question: str
    search_query: str
    answer: str
```

### Node 1: rewrite question

```python
def rewrite_query(state: State):
    return {
        "search_query": f"best answer for: {state['question']}"
    }
```

### Node 2: generate answer

```python
def answer_node(state: State):
    return {
        "answer": f"Using search query: {state['search_query']}"
    }
```

### Flow

```text
START → rewrite_query → answer_node → END
```

### Execution

Initial state:

```python
{
    "question": "What is AI?",
    "search_query": "",
    "answer": ""
}
```

After `rewrite_query`:

```python
{
    "question": "What is AI?",
    "search_query": "best answer for: What is AI?",
    "answer": ""
}
```

After `answer_node`:

```python
{
    "question": "What is AI?",
    "search_query": "best answer for: What is AI?",
    "answer": "Using search query: best answer for: What is AI?"
}
```

This is why state is the **backbone of graph communication**.

---

# 10) Common Things Stored in State

In real LangGraph apps, state often includes fields like:

```python
class State(TypedDict):
    user_input: str
    messages: list
    retrieved_docs: list
    tool_result: str
    answer: str
    error: str
    step_count: int
    should_search: bool
```

Typical categories:

## Input fields

* user question
* user message
* uploaded content
* system instructions

## Intermediate fields

* rewritten query
* tool outputs
* retrieved documents
* classification labels
* planning steps

## Final output fields

* answer
* summary
* structured response

## Control fields

* retry count
* route decision
* error flag
* whether tool use is needed

---

# 11) Supported Ways to Define State

LangGraph supports multiple state schemas.

## Option A — `TypedDict` (**most common / recommended**)

```python
from typing_extensions import TypedDict

class State(TypedDict):
    question: str
    answer: str
```

### Why it’s commonly recommended

* simple
* lightweight
* readable
* good for most graphs
* works well with Python typing

---

## Option B — `dataclass`

```python
from dataclasses import dataclass

@dataclass
class State:
    question: str
    answer: str
```

Useful if you prefer dataclass style.

---

## Option C — Pydantic model

```python
from pydantic import BaseModel

class State(BaseModel):
    question: str
    answer: str
```

Useful when you want:

* validation
* stricter schemas
* richer model behavior

But it can be slower than `TypedDict`.

---

# 12) Which One Should You Use?

My practical recommendation for learning order:

## Use `TypedDict` first

Because it makes the LangGraph model easiest to understand.

Start with:

```python
from typing_extensions import TypedDict

class State(TypedDict):
    input: str
    output: str
```

Then later, if needed:

* move to `Pydantic` for validation
* use more advanced state shapes
* add reducers for list accumulation and message history

---

# 13) State Schema = Contract for the Graph

Your state definition acts like a **contract** for all nodes.

If state is:

```python
class State(TypedDict):
    question: str
    answer: str
```

then all nodes should work with those keys correctly.

For example:

```python
def node_a(state: State):
    return {"answer": "done"}
```

works.

But this is dangerous:

```python
def node_a(state: State):
    return {"result": "done"}
```

because `result` is **not part of the defined state schema**.

So the schema helps keep graph logic predictable.

---

# 14) State Before and After Each Node

A good way to think:

## Before node runs

State = **input to node**

## After node returns

Returned dict = **update patch**

## After LangGraph merges it

New state = **updated graph state**

Formula:

```text
new_state = old_state + node_update
```

Not literally Python addition — conceptually a merge/update.

In [2]:
# 15) Example with Multiple State Keys
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    user_input: str
    category: str
    answer: str

def classify(state: State):
    text = state["user_input"].lower()

    if "python" in text:
        return {"category": "programming"}
    return {"category": "general"}

def respond(state: State):
    if state["category"] == "programming":
        return {"answer": "This looks like a programming question."}
    return {"answer": "This looks like a general question."}

builder = StateGraph(State)
builder.add_node("classify", classify)
builder.add_node("respond", respond)

builder.add_edge(START, "classify")
builder.add_edge("classify", "respond")
builder.add_edge("respond", END)

graph = builder.compile()

result = graph.invoke({
    "user_input": "How do I use Python lists?",
    "category": "",
    "answer": ""
})

print(result)

{'user_input': 'How do I use Python lists?', 'category': 'programming', 'answer': 'This looks like a programming question.'}


### Final output

```python
{
    "user_input": "How do I use Python lists?",
    "category": "programming",
    "answer": "This looks like a programming question."
}
```

---

# 16) State Is Not Just “Input”; It Is the Whole Working Memory

This distinction matters.

A beginner often thinks:

> “State is just the input I pass to `invoke()`.”

Not quite.

State is:

* the initial input **plus**
* all intermediate values created during execution **plus**
* final outputs stored by nodes

So state is really the **working memory of the graph**.

---

# 17) Realistic LLM Example State

Here’s a more practical schema for an AI workflow:

```python
from typing_extensions import TypedDict

class State(TypedDict):
    question: str
    rewritten_question: str
    retrieved_context: str
    answer: str
```

Possible graph:

```text
START
  ↓
rewrite_question
  ↓
retrieve_docs
  ↓
generate_answer
  ↓
END
```

### Node responsibilities

## `rewrite_question`

reads `question`, writes `rewritten_question`

## `retrieve_docs`

reads `rewritten_question`, writes `retrieved_context`

## `generate_answer`

reads `question + retrieved_context`, writes `answer`

This is exactly how LangGraph scales from tiny demos to agentic systems.

---

# 18) A Very Important Design Skill: Choosing Good State Keys

A strong LangGraph design depends heavily on **clean state design**.

## Bad state design

```python
class State(TypedDict):
    data: str
```

Why bad?
Because `data` is too vague.
Does it mean question? docs? answer? tool output?

---

## Better state design

```python
class State(TypedDict):
    user_question: str
    retrieved_docs: list[str]
    tool_result: str
    final_answer: str
```

Much better because each key has a clear job.

### Good rule:

Make state keys **specific and intentional**.

---

# 19) Common Beginner Mistakes with State

## Mistake 1: forgetting to include a field in the state schema

Example:

```python
class State(TypedDict):
    question: str
```

but a node returns:

```python
return {"answer": "hello"}
```

If your graph depends on `answer`, it should be part of the schema.

---

## Mistake 2: returning the whole state every time

You usually only want to return updated keys.

---

## Mistake 3: using unclear field names

Avoid generic names like:

* `data`
* `value`
* `result`
* `info`

unless the meaning is obvious.

---

## Mistake 4: mixing permanent state with runtime-only dependencies

Things like:

* DB connection
* model client
* external service object

should usually go into **runtime context**, not graph state.

State should hold **workflow data**, not heavy external objects.

---

# 20) State vs Runtime Context

This is a very important distinction.

## State

Persistent workflow data that nodes read/update

Examples:

* question
* answer
* messages
* docs
* retry count

## Runtime context

External config/dependencies not meant to be part of graph data

Examples:

* LLM client
* DB connection
* environment config
* API clients
* provider settings

### Simple rule:

If it’s part of the **workflow’s evolving data**, it belongs in **state**.
If it’s a **supporting dependency/config object**, it belongs in **runtime context**.

---

# 21) Mini Example: State + Runtime Context

```python
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    question: str
    answer: str

def answer_node(state: State, runtime):
    provider = runtime.context.get("provider", "unknown")

    return {
        "answer": f"Provider: {provider} | Question: {state['question']}"
    }
```

Here:

## State contains

* question
* answer

## Runtime context contains

* provider configuration

That separation is clean and scalable.

---

# 22) Visual Flow of State

```text
Initial State
{
  "question": "What is LangGraph?",
  "answer": ""
}

        ↓

Node A reads question
Node A returns {"answer": "LangGraph is ..."}

        ↓

Merged State
{
  "question": "What is LangGraph?",
  "answer": "LangGraph is ..."
}
```

---

# 23) Practical Rule for Designing State in Real Projects

When building a LangGraph app, define state by asking:

## “What information must survive from one node to the next?”

Whatever must survive across node boundaries should usually be in **state**.

Example for a RAG graph:

```python
class State(TypedDict):
    user_question: str
    rewritten_query: str
    retrieved_chunks: list[str]
    final_answer: str
```

Because each of those values is needed later by another node.

---

# 24) Interview-Level Summary of State

If asked in an interview:

## **What is State in LangGraph?**

State is the shared data structure that flows through the graph.
Each node reads the current state and returns updates to it. LangGraph merges those updates and passes the updated state to subsequent nodes. It acts as the graph’s working memory and is typically defined using `TypedDict`, `dataclass`, or `Pydantic`.

---

# 25) Quick Revision Notes

## State in one line

**State = shared memory for the graph**

## Nodes do what with state?

* read it
* compute something
* return updates

## State stores

* inputs
* intermediate results
* final outputs
* control flags

## Common schema choices

* `TypedDict` ✅
* `dataclass`
* `Pydantic`

## Best practice

* use clear key names
* return only changed fields
* keep runtime dependencies out of state

---

# 26) Full Study Cheat Sheet — State

```text
STATE IN LANGGRAPH

Definition
----------
State is the shared data object that moves through the graph.

What it does
------------
- Stores graph data
- Passes data from one node to another
- Keeps intermediate results
- Holds final outputs

Node behavior
-------------
Node receives current state
Node returns state updates
LangGraph merges updates into state

Typical contents
----------------
- user input
- rewritten query
- tool outputs
- retrieved docs
- final answer
- counters / flags / errors

Common schema choices
---------------------
1. TypedDict  ← recommended for learning
2. dataclass
3. Pydantic BaseModel

Best practices
--------------
- Keep field names explicit
- Return only updated keys
- Put workflow data in state
- Put clients/config in runtime context
```

---

# 27) Practice Questions on State

1. What is the role of state in LangGraph?
2. Why is state called shared memory?
3. What does a node receive as input?
4. What should a node return: full state or state updates?
5. Why is `TypedDict` commonly used for state?
6. What kind of data belongs in state?
7. What should go into runtime context instead of state?
8. How does state allow nodes to communicate?

---

# 28) Mini Assignment for You

Try building this yourself:

## Task

Create a graph with state:

```python
class State(TypedDict):
    text: str
    word_count: int
    result: str
```

### Node 1

Count words in `text` and store in `word_count`

### Node 2

Create a message like:

```python
"This text has X words"
```

and store it in `result`

### Flow

```text
START → count_words → make_result → END
```


In [5]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    text: str
    word_count: int
    result: str

# Node 1: count words
def count_words(state: State):
    words = state["text"].split()
    return {"word_count": len(words)}


# Node 2: create result message
def make_result(state: State):
    return {"result": f"This text has {state['word_count']} words"}

# Build graph
graph_builder = StateGraph(State)

graph_builder.add_node("count_words", count_words)
graph_builder.add_node("make_result", make_result)

graph_builder.add_edge(START, "count_words")
graph_builder.add_edge("count_words", "make_result")
graph_builder.add_edge("make_result", END)

graph = graph_builder.compile()


# Run graph
output = graph.invoke({"text": "LangGraph makes workflows easy"})
print(output)


{'text': 'LangGraph makes workflows easy', 'word_count': 4, 'result': 'This text has 4 words'}
